<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/All_Buried_Vol_Angles_Stericmaps_Gamma_Arylation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# INSTALL
# ============================================
!pip install morfeus-ml seaborn openpyxl

# ============================================
# IMPORTS
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from morfeus import (
    BuriedVolume,
    ConeAngle,
    read_xyz
)

# ============================================
# PATHS
# ============================================
xyz_folder = "/content/drive/MyDrive/xyz_files"

plot_folder = (
    "/content/drive/MyDrive/Gamma_Arylation_Plots"
)

os.makedirs(plot_folder, exist_ok=True)

excel_file = (
    "/content/drive/MyDrive/Gamma_Arylation_Sterics.xlsx"
)

# ============================================
# YIELDS
# ============================================
yield_dict = {

"L7":0,
"L8":0,
"L9":0,
"L10":0,
"L11":0,
"L12":0,
"L13":0,
"L14":0,
"L15":0,

"L16":12,
"L17":53,
"L18":46,
"L19":67,
"L20":40,
"L21":31,

"L22":0,
"L23":0,
"L24":0,
"L25":0
}

# ============================================
# METAL
# ============================================
metal_atom = 1

# ============================================
# STORAGE
# ============================================
results = []

print("Processing ligands...\n")

# ============================================
# LOOP
# ============================================
for file in sorted(os.listdir(xyz_folder)):

    if not file.endswith(".xyz"):
        continue

    try:

        file_path = os.path.join(
            xyz_folder,
            file
        )

        elements, coordinates = read_xyz(
            file_path
        )

        # --------------------------------
        # Ligand number
        # --------------------------------
        match = re.search(
            r"L(\d+)",
            file
        )

        if not match:
            continue

        lig_num = int(
            match.group(1)
        )

        ligand_name = f"L{lig_num}"

        # --------------------------------
        # Ligand atoms
        # --------------------------------
        ligand_atoms = list(
            range(
                17,
                len(elements)+1
            )
        )

        all_atoms = list(
            range(
                1,
                len(elements)+1
            )
        )

        excluded_atoms = [
            i for i in all_atoms
            if i not in ligand_atoms
        ]

        # --------------------------------
        # Buried Volume
        # --------------------------------
        bv = BuriedVolume(
            elements,
            coordinates,
            metal_atom,
            excluded_atoms=excluded_atoms
        )

        vbur = (
            bv.fraction_buried_volume
            * 100
        )

        # --------------------------------
        # Distal Volume
        # --------------------------------
        try:

            bv.compute_distal_volume()

            distal = (
                bv.distal_volume
            )

        except:

            distal = None

        # --------------------------------
        # Cone Angle
        # --------------------------------
        try:

            keep_atoms = (
                [1]
                + ligand_atoms
            )

            lig_elements = [
                elements[i-1]
                for i in keep_atoms
            ]

            lig_coords = [
                coordinates[i-1]
                for i in keep_atoms
            ]

            ca = ConeAngle(
                lig_elements,
                lig_coords,
                1
            )

            cone = (
                ca.cone_angle
            )

        except:

            cone = None

        # --------------------------------
        # Save
        # --------------------------------
        results.append({

            "Ligand":
                ligand_name,

            "Vbur_ligand (%)":
                vbur,

            "Distal Volume (A3)":
                distal,

            "Cone Angle (deg)":
                cone,

            "Gamma_Arylation_Yield (%)":
                yield_dict.get(
                    ligand_name,
                    0
                )

        })

        print(
            f"Done: {ligand_name}"
        )

    except Exception as e:

        print(
            f"Error in {file}: {e}"
        )

# ============================================
# DATAFRAME
# ============================================
df = pd.DataFrame(results)

# ============================================
# SAVE EXCEL
# ============================================
df.to_excel(
    excel_file,
    index=False
)

print("\nExcel saved:")
print(excel_file)

# ============================================
# YIELD vs DISTAL
# ============================================
plt.figure(
    figsize=(5,4),
    dpi=300
)

plt.scatter(
    df["Distal Volume (A3)"],
    df["Gamma_Arylation_Yield (%)"],
    s=100
)

for _, r in df.iterrows():

    plt.text(
        r["Distal Volume (A3)"],
        r["Gamma_Arylation_Yield (%)"],
        r["Ligand"],
        fontsize=8
    )

plt.xlabel(
    "Distal Volume (A³)"
)

plt.ylabel(
    "Yield (%)"
)

plt.tight_layout()

plt.savefig(
    f"{plot_folder}/Yield_vs_Distal.jpg",
    dpi=600
)

plt.close()

# ============================================
# YIELD vs CONE ANGLE
# ============================================
plt.figure(
    figsize=(5,4),
    dpi=300
)

plt.scatter(
    df["Cone Angle (deg)"],
    df["Gamma_Arylation_Yield (%)"],
    s=100
)

for _, r in df.iterrows():

    plt.text(
        r["Cone Angle (deg)"],
        r["Gamma_Arylation_Yield (%)"],
        r["Ligand"],
        fontsize=8
    )

plt.xlabel(
    "Cone Angle (deg)"
)

plt.ylabel(
    "Yield (%)"
)

plt.tight_layout()

plt.savefig(
    f"{plot_folder}/Yield_vs_Cone.jpg",
    dpi=600
)

plt.close()

# ============================================
# STERIC MAP
# ============================================
plt.figure(
    figsize=(6,5),
    dpi=300
)

sc = plt.scatter(
    df["Distal Volume (A3)"],
    df["Vbur_ligand (%)"],
    c=df[
        "Gamma_Arylation_Yield (%)"
    ],
    s=150
)

for _, r in df.iterrows():

    plt.text(
        r["Distal Volume (A3)"],
        r["Vbur_ligand (%)"],
        r["Ligand"],
        fontsize=8
    )

plt.colorbar(
    sc,
    label="Yield (%)"
)

plt.xlabel(
    "Distal Volume (A³)"
)

plt.ylabel(
    "%Vbur"
)

plt.tight_layout()

plt.savefig(
    f"{plot_folder}/Steric_Map.jpg",
    dpi=600
)

plt.close()

# ============================================
# HEATMAP
# ============================================
corr = df[[
    "Vbur_ligand (%)",
    "Distal Volume (A3)",
    "Cone Angle (deg)",
    "Gamma_Arylation_Yield (%)"
]].corr()

plt.figure(
    figsize=(6,5),
    dpi=300
)

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    square=True
)

plt.tight_layout()

plt.savefig(
    f"{plot_folder}/Correlation_Heatmap.jpg",
    dpi=600
)

plt.close()

print("\nAll plots generated.")
print(plot_folder)

Mounted at /content/drive
Processing ligands...



/tmp/ipykernel_1411/1622816243.py:187: UserWarning: Failed to import libconeangle. Defaulting to method='internal'
  ca = ConeAngle(


Done: L10
Done: L11
Done: L12
Done: L13
Done: L14
Done: L15
Done: L16
Done: L17
Done: L18
Done: L19
Done: L20
Done: L21
Done: L22
Done: L23
Done: L24
Done: L25
Done: L7
Done: L8
Done: L9

Excel saved:
/content/drive/MyDrive/Gamma_Arylation_Sterics.xlsx

All plots generated.
/content/drive/MyDrive/Gamma_Arylation_Plots
